# Unfold studied MC with a controlled-sample response

Build a flattened $p_T^{ave}$–$\eta_{CM}$ response from the approximately 60% controlled MC subsample, unfold the independent approximately 40% studied reconstructed distribution, and compare the result with studied generator truth. The measured momentum belongs to reconstructed AK4 jets and includes the data-driven default JER smearing. Only controlled-sample objects construct the response, inefficiency, and fake correction.

## Analysis model and closure logic

Each pair $(p_T^{ave}$ interval, $\eta_{CM}$ bin$)$ is flattened into one global index. This preserves the full two-dimensional migration: an off-diagonal matrix element may change eta, $p_T^{ave}$, or both. For the controlled response,

$$P_i^{ctrl}=m_i^{matched,ctrl}/m_i^{all,ctrl}, \qquad \epsilon_j^{ctrl}=t_j^{matched,ctrl}/t_j^{all,ctrl}.$$

The studied estimate is

$$\hat t_j^{studied}=\frac{1}{\epsilon_j^{ctrl}}\,U_j\!\left[P_i^{ctrl}m_i^{all,studied};M^{ctrl}\right],$$

where $U$ is iterative Bayesian unfolding using only the matched controlled migration matrix. This is valid only to the extent that controlled and studied subsamples share the same detector response, purity, and efficiency. Unlike same-sample closure, statistical fluctuations in two independent samples do not cancel, so individual ratios need not equal one. More iterations reduce prior dependence but amplify statistical variation; convergence is not a promise that every bin approaches unity.

Forward folding tests the reco-space consequence of the unfolded result. The inclusive controlled response maps $\hat t^{studied}$ to matched reco, including inefficiency through its column normalization. Fakes are then restored bin by bin using $(m_i^{fake}/m_i^{matched})_{ctrl}$. The result is compared with inclusive studied reco. This closure can disagree with unity because of independent fluctuations or controlled-to-studied response differences even when the implementation is correct.

<!-- detailed-workflow-guide -->

### Detailed workflow and inverse problem

The pair $(p_T^{ave},\eta_{CM})$ is mapped to a global bin so the full response retains migrations in both coordinates. With $M_{ij}$ denoting matched events from truth bin $j$ to reco bin $i$, inclusive marginals satisfy $t_j=t_j^{matched}+t_j^{miss}$ and $m_i=m_i^{matched}+m_i^{fake}$.

Factorized unfolding applies $m_i^{signal}=P_im_i$ with $P_i=m_i^{matched}/m_i$, unfolds matched migrations with iterative Bayes, and returns inclusive truth through $\hat t_j=\hat t_j^{matched}/\epsilon_j$, $\epsilon_j=t_j^{matched}/t_j$. Forward folding reverses the physical mapping: the inclusive response reapplies efficiency and migration, then fakes are restored from the training fake-to-matched-reco ratio. Iterations regulate prior dependence versus variance; they do not guarantee unity for independent samples.

In [ ]:
# Cell role: initialize the reproducible Python/ROOT environment and shared helpers.
# Interpretation: No physics histogram is modified here; ROOT ownership is configured before files open.
# The preceding Markdown gives the equations and physics assumptions for this step.
%load_ext autoreload
%autoreload 2

from pathlib import Path
import os
import sys

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / 'CMakeLists.txt').is_file()
                     and (p / 'hist_analysis').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Start Jupyter from the jetAnalysis repository root')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hist_analysis.python.notebook_setup import load_root, load_roounfold
ROOT = load_root(batch=True)
from hist_analysis.config.files import BASE_DIR
from hist_analysis.config.histograms import DIJET_DELTA_PHI_SELECTION_LABEL
from hist_analysis.python.histogram_io import (
    load_histogram, resolve_combined_file, resolve_direction_file,
)
ROOUNFOLD_ROOT, ROOUNFOLD_LIBRARY = load_roounfold(ROOT, project_root=PROJECT_ROOT)
import RooUnfold
ROOT.gStyle.SetOptStat(0)
ROOT.gStyle.SetPalette(ROOT.kBird)
ROOT.TH1.AddDirectory(False)

from hist_analysis.config.histograms import DIJET_PTAVE_BINS, TEST_DIJET_PTAVE_BINS
from hist_analysis.python.unfolding import (
    UnfoldingInputKeys, apply_efficiency_correction, as_pt_intervals,
    build_roounfold_response, calculate_closure_metrics,
    calculate_response_diagnostics, flatten_pt_eta_projections,
    flatten_sparse_response, forward_fold_truth, load_unfolding_inputs,
    prepare_factorized_corrections, project_eta_by_pt, unfold_bayes,
    unflatten_to_eta_projections,
    validate_response_accounting, write_unfolding_output,
)
from hist_analysis.python.unfolding_plots import (
    draw_controlled_studied_shape_comparisons, draw_flattened_response,
    draw_response_components, draw_response_matrix,
    draw_unfolding_closure, draw_unfolding_closure_by_pt,
    range_covering_histogram_bins,
)
from hist_analysis.python.plotting import draw_closure


## Configuration

The response-training objects all come from the controlled sample. The measured spectrum and closure truth both come from the studied sample. Intervals are half-open. `RESPONSE_SCALE` protects small weighted response entries from RooUnfold's absolute sanitization threshold.


In [ ]:
# Cell role: define and validate user-facing analysis configuration.
# Interpretation: Changing these values can change inputs, selections, binning, normalization, or outputs.
# The preceding Markdown gives the equations and physics assumptions for this step.
GENERATOR = 'embedding'       # embedding or pythia
DIRECTION = 'Pbgoing'        # pgoing, Pbgoing, or combined
FILE_STEM = 'jetId'
ETA_CUTS = (1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.3, 2.4, 3.0)
ETA_CUT_INDEX = 5
PTAVE_BIN_SET = 'standard'  # test is only a quick execution check
PTAVE_BIN_SETS = {'test': TEST_DIJET_PTAVE_BINS, 'standard': DIJET_PTAVE_BINS}
PT_AVE_BINS = tuple(PTAVE_BIN_SETS[PTAVE_BIN_SET])
N_ITERATIONS = 4
REQUIRE_FAKES = True
HANDLE_FAKES = True
FORWARD_FOLD_ADD_FAKES = True
RESPONSE_SCALE = 1.0e12
PLOT_MISS_AND_FAKES = False
RATIO_Y_RANGE = (0.75, 1.25)
DRAW_GRID = True
SAVE_PNG = False

CONTROLLED_TRUTH_TEMPLATE = 'hGenDijetPtEtaCMControlledSample_{eta_cut_index}'
CONTROLLED_MEASURED_TEMPLATE = 'hRecoDijetPtEtaCMJerDefUnfoldControlledSample_{eta_cut_index}'
STUDIED_TRUTH_TEMPLATE = 'hGenDijetPtEtaCMStudiedSample_{eta_cut_index}'
STUDIED_MEASURED_TEMPLATE = 'hRecoDijetPtEtaCMJerDefUnfoldStudiedSample_{eta_cut_index}'
CONTROLLED_RESPONSE_TEMPLATE = 'hGenDijetPtEtaCMVsRecoPtEtaCMControlledSample_{eta_cut_index}'
CONTROLLED_MISS_TEMPLATE = 'hGenDijetPtEtaCMMissControlledSample_{eta_cut_index}'
CONTROLLED_FAKE_TEMPLATE = 'hRecoDijetPtEtaCMFakeControlledSample_{eta_cut_index}'

if PTAVE_BIN_SET not in PTAVE_BIN_SETS:
    raise ValueError(f'Unsupported PTAVE_BIN_SET={PTAVE_BIN_SET!r}')
if GENERATOR not in ('embedding', 'pythia'):
    raise ValueError(f'Unsupported GENERATOR={GENERATOR!r}')
if DIRECTION not in ('pgoing', 'Pbgoing', 'combined'):
    raise ValueError(f'Unsupported DIRECTION={DIRECTION!r}')
if ETA_CUT_INDEX < 0 or ETA_CUT_INDEX >= len(ETA_CUTS):
    raise IndexError(f'Invalid ETA_CUT_INDEX={ETA_CUT_INDEX}')
if RESPONSE_SCALE <= 0.0:
    raise ValueError('RESPONSE_SCALE must be positive')
if not isinstance(PLOT_MISS_AND_FAKES, bool):
    raise TypeError('PLOT_MISS_AND_FAKES must be True or False')
if not all(isinstance(value, bool) for value in (REQUIRE_FAKES, HANDLE_FAKES, FORWARD_FOLD_ADD_FAKES)):
    raise TypeError('REQUIRE_FAKES, HANDLE_FAKES, and FORWARD_FOLD_ADD_FAKES must be booleans')
if len(RATIO_Y_RANGE) != 2 or RATIO_Y_RANGE[0] >= RATIO_Y_RANGE[1]:
    raise ValueError('RATIO_Y_RANGE must contain two increasing values')
if PTAVE_BIN_SET == 'test':
    print('WARNING: test pT bins contain gaps and are intended only for quick checks')

if DIRECTION == 'combined':
    INPUT_FILE = resolve_combined_file(BASE_DIR, GENERATOR, FILE_STEM)
else:
    INPUT_FILE = resolve_direction_file(BASE_DIR, GENERATOR, DIRECTION, FILE_STEM)
ETA_CUT = ETA_CUTS[ETA_CUT_INDEX]
ETA_UNFOLDING_RANGE = (-ETA_CUT, ETA_CUT)
ETA_TAG = f'{ETA_CUT:g}'.replace('.', 'p')
OUTPUT_DIR = Path(os.environ.get(
    'DIJET_UNFOLD2D_STUDIED_CONTROLLED_OUTPUT_DIR',
    PROJECT_ROOT / 'hist_analysis' / 'output' / 'unfold2D_studied_to_controlled',
))
OUTPUT_BASE_TAG = (f'{GENERATOR}_{DIRECTION}_unfold2D_studiedToControlled_'
                   f'eta_{ETA_TAG}')
OUTPUT_TAG = f'{OUTPUT_BASE_TAG}_iter_{N_ITERATIONS}'
OUTPUT_ROOT_FILE = OUTPUT_DIR / f'{OUTPUT_TAG}.root'

PT_AVE_INTERVALS = as_pt_intervals(PT_AVE_BINS)


## Load controlled training objects and studied closure objects


In [ ]:
# Cell role: resolve input files and load the named ROOT objects.
# Interpretation: Loaded objects that outlive their file must be cloned and detached from ROOT directories.
# The preceding Markdown gives the equations and physics assumptions for this step.
controlled_keys = UnfoldingInputKeys(
    truth=CONTROLLED_TRUTH_TEMPLATE.format(eta_cut_index=ETA_CUT_INDEX),
    measured=CONTROLLED_MEASURED_TEMPLATE.format(eta_cut_index=ETA_CUT_INDEX),
    response=CONTROLLED_RESPONSE_TEMPLATE.format(eta_cut_index=ETA_CUT_INDEX),
    miss=CONTROLLED_MISS_TEMPLATE.format(eta_cut_index=ETA_CUT_INDEX),
    fake=CONTROLLED_FAKE_TEMPLATE.format(eta_cut_index=ETA_CUT_INDEX),
)
controlled_inputs = load_unfolding_inputs(INPUT_FILE, controlled_keys)
controlled_truth_2d = controlled_inputs.truth
controlled_reco_2d = controlled_inputs.measured
controlled_response_sparse = controlled_inputs.response
controlled_explicit_miss_2d = controlled_inputs.miss
controlled_explicit_fake_2d = controlled_inputs.fake
studied_truth_2d = load_histogram(
    str(INPUT_FILE),
    STUDIED_TRUTH_TEMPLATE.format(eta_cut_index=ETA_CUT_INDEX),
)
studied_reco_2d = load_histogram(
    str(INPUT_FILE),
    STUDIED_MEASURED_TEMPLATE.format(eta_cut_index=ETA_CUT_INDEX),
)
print(f'Input: {INPUT_FILE}')
print(controlled_keys)

## Project pTave intervals and construct flattened inputs

Global bins are ordered as `[pTave block][eta bin]`. All response pT migration blocks are retained.


In [ ]:
# Cell role: project multidimensional inputs into the configured analysis bins.
# Interpretation: Selections are applied before arithmetic so migrations and boundary bins remain explicit.
# The preceding Markdown gives the equations and physics assumptions for this step.
projection_options = {
    'pt_bins': PT_AVE_INTERVALS,
    'eta_range': ETA_UNFOLDING_RANGE,
}
controlled_truth_by_pt = project_eta_by_pt(
    controlled_truth_2d, name_prefix='hControlledTruthEtaCM', **projection_options)
controlled_reco_by_pt = project_eta_by_pt(
    controlled_reco_2d, name_prefix='hControlledRecoEtaCM', **projection_options)
studied_truth_by_pt = project_eta_by_pt(
    studied_truth_2d, name_prefix='hStudiedTruthEtaCM', **projection_options)
studied_reco_by_pt = project_eta_by_pt(
    studied_reco_2d, name_prefix='hStudiedRecoEtaCM', **projection_options)
controlled_explicit_miss_by_pt = project_eta_by_pt(
    controlled_explicit_miss_2d, name_prefix='hControlledExplicitMissEtaCM',
    **projection_options)
controlled_explicit_fake_by_pt = project_eta_by_pt(
    controlled_explicit_fake_2d, name_prefix='hControlledExplicitFakeEtaCM',
    **projection_options)

controlled_truth, layout = flatten_pt_eta_projections(
    controlled_truth_by_pt, name='hControlledTruthEtaCM', pt_bins=PT_AVE_INTERVALS)
controlled_reco, _ = flatten_pt_eta_projections(
    controlled_reco_by_pt, name='hControlledRecoEtaCM', layout=layout)
studied_truth, _ = flatten_pt_eta_projections(
    studied_truth_by_pt, name='hStudiedTruthEtaCM', layout=layout)
studied_reco, _ = flatten_pt_eta_projections(
    studied_reco_by_pt, name='hStudiedRecoEtaCM', layout=layout)
controlled_explicit_miss, _ = flatten_pt_eta_projections(
    controlled_explicit_miss_by_pt, name='hControlledExplicitMissEtaCM', layout=layout)
controlled_explicit_fake, _ = flatten_pt_eta_projections(
    controlled_explicit_fake_by_pt, name='hControlledExplicitFakeEtaCM', layout=layout)
controlled_response, _ = flatten_sparse_response(
    controlled_response_sparse, PT_AVE_INTERVALS,
    name='hControlledResponseEtaCM', layout=layout,
    eta_range=ETA_UNFOLDING_RANGE)
print(
    f'pT blocks={layout.n_pt_bins}, eta bins={layout.n_eta_bins}, '
    f'global bins={layout.n_global_bins}'
)

## Original controlled and studied shapes

Compare the original Gen shapes and the original reconstructed shapes before unfolding. Every histogram is cloned and independently normalized as $1/N dN/d\eta_{CM}$ using its bin widths, so these plots test shape compatibility without changing the response inputs or retaining the intentional 60/40 yield difference. The lower panels show the studied/controlled ratios of those normalized shapes.


In [ ]:
# Cell role: construct derived ratios, efficiencies, or correction factors.
# Interpretation: The numerator/denominator relationship determines whether independent or binomial errors are valid.
# The preceding Markdown gives the equations and physics assumptions for this step.
(original_shape_canvases, normalized_original_shapes,
 original_studied_to_controlled_ratios) = (
    draw_controlled_studied_shape_comparisons(
        controlled_truth_by_pt, studied_truth_by_pt,
        controlled_reco_by_pt, studied_reco_by_pt,
        PT_AVE_INTERVALS, output_dir=OUTPUT_DIR, output_tag=OUTPUT_BASE_TAG,
        eta_range=ETA_UNFOLDING_RANGE,
        ratio_range=RATIO_Y_RANGE,
        annotations=(f'|#eta_{{CM}}| < {ETA_CUT:g}',),
        save_png=SAVE_PNG, grid=DRAW_GRID,
    )
)
original_shape_canvases


## Validate the controlled response and unfold studied reco

Before constructing RooUnfold, verify that the controlled truth and reco marginals equal the matched response projections plus the effective miss and fake components. Then multiply studied reco by the controlled reco-bin purity, unfold with the controlled matched-event migration matrix, and divide by the controlled truth-bin efficiency: $t_j=(U[P\,m])_j/\epsilon_j$. The conventional RooUnfold fake/miss treatment is retained as a reference. The Bayesian iteration count remains fixed by the configuration.


In [ ]:
# Cell role: project multidimensional inputs into the configured analysis bins.
# Interpretation: Selections are applied before arithmetic so migrations and boundary bins remain explicit.
# The preceding Markdown gives the equations and physics assumptions for this step.
response_diagnostics = calculate_response_diagnostics(
    controlled_response, controlled_truth, controlled_reco,
    explicit_miss=controlled_explicit_miss,
    explicit_fake=controlled_explicit_fake,
)
response_accounting = validate_response_accounting(
    controlled_response, controlled_truth, controlled_reco,
    response_diagnostics,
)
print('Controlled response accounting:')
print(f'  max truth residual: {response_accounting.max_truth_residual:.3g}')
print(f'  max measured residual: {response_accounting.max_measured_residual:.3g}')
print(f'  global efficiency: {response_accounting.global_efficiency:.4f}')
print(f'  global fake fraction: {response_accounting.global_fake_fraction:.4f}')
print(f'  response sparsity: {response_accounting.response_sparsity:.4f}')
print(f'  zero-efficiency truth bins: {response_accounting.zero_efficiency_truth_bins}')
print(f'  empty measured bins: {response_accounting.empty_measured_bins}')

# Reference calculation: RooUnfold handles the effective fakes and misses.
inclusive_response_bundle = build_roounfold_response(
    RooUnfold, controlled_truth, controlled_reco, controlled_response,
    diagnostics=response_diagnostics, scale=RESPONSE_SCALE,
    require_fakes=REQUIRE_FAKES, name='controlledResponseEtaCM',
    title='Controlled-sample flattened dijet response',
)
inclusive_unfolding_result = unfold_bayes(
    RooUnfold, inclusive_response_bundle, studied_reco, iterations=N_ITERATIONS,
    handle_fakes=HANDLE_FAKES, name='hUnfoldedStudiedInclusiveResponseEtaCM',
)
unfolded_studied_inclusive_response = inclusive_unfolding_result.histogram

# Primary calculation: controlled purity -> matched migration -> controlled efficiency.
factorized_inputs = prepare_factorized_corrections(
    studied_reco, controlled_reco, response_diagnostics.matched_measured,
    controlled_truth, response_diagnostics.matched_truth,
    name_prefix='hStudiedFactorized',
)
studied_reco_signal = factorized_inputs.measured_signal
controlled_reco_purity = factorized_inputs.purity
controlled_truth_efficiency = factorized_inputs.efficiency
matched_response_bundle = build_roounfold_response(
    RooUnfold, response_diagnostics.matched_truth,
    response_diagnostics.matched_measured, controlled_response,
    scale=RESPONSE_SCALE, require_fakes=False,
    name='controlledMatchedResponseEtaCM',
    title='Controlled-sample matched-event dijet response',
)
matched_unfolding_result = unfold_bayes(
    RooUnfold, matched_response_bundle, studied_reco_signal,
    iterations=N_ITERATIONS, handle_fakes=False,
    name='hUnfoldedStudiedMatchedEtaCM',
)
unfolded_studied_matched = matched_unfolding_result.histogram
unfolded_studied, covariance = apply_efficiency_correction(
    unfolded_studied_matched, matched_unfolding_result.covariance,
    controlled_truth_efficiency, name='hUnfoldedStudiedEtaCM',
)
response_bundle = matched_response_bundle

controlled_truth_fraction = controlled_truth.Integral() / (
    controlled_truth.Integral() + studied_truth.Integral())
controlled_reco_fraction = controlled_reco.Integral() / (
    controlled_reco.Integral() + studied_reco.Integral())
print(f'Controlled weighted truth fraction: {controlled_truth_fraction:.4f}')
print(f'Controlled weighted reco fraction: {controlled_reco_fraction:.4f}')
print(f'Fixed Bayesian iterations: {N_ITERATIONS}')

## Separate response-component diagnostics and studied closure

The controlled response is shown first. A separate canvas decomposes explicit selection/matching failures, pT-boundary migrations, and the effective miss/fake spectra used by RooUnfold. No studied response, miss, or fake object enters response construction.


In [ ]:
# Cell role: project multidimensional inputs into the configured analysis bins.
# Interpretation: Selections are applied before arithmetic so migrations and boundary bins remain explicit.
# The preceding Markdown gives the equations and physics assumptions for this step.
response_canvas = draw_flattened_response(
    {
        'gen': ('Controlled Gen', controlled_truth),
        'reco': ('Controlled reco', controlled_reco),
        'miss': ('Controlled explicit miss', controlled_explicit_miss),
        'fake': ('Controlled explicit fake', controlled_explicit_fake),
    },
    controlled_response,
    annotations=(f'Controlled sample, |#eta_{{CM}}| < {ETA_CUT:g}',),
    plot_miss_and_fakes=PLOT_MISS_AND_FAKES,
    output=OUTPUT_DIR / f'{OUTPUT_TAG}_flattened_response.pdf',
    save_png=SAVE_PNG, grid=DRAW_GRID,
    canvas_name='canvas_controlled_response',
)
response_matrix_canvas = draw_response_matrix(
    controlled_response,
    annotations=(
        'Controlled sample only',
        f'|#eta_{{CM}}| < {ETA_CUT:g}',
        f'{layout.n_pt_bins} p_{{T}}^{{ave}} intervals',
    ),
    output=OUTPUT_DIR / f'{OUTPUT_TAG}_response_matrix.pdf',
    save_png=SAVE_PNG, grid=False, log_z=True,
    canvas_name='canvas_controlled_response_matrix',
)
response_components_canvas = draw_response_components(
    explicit_miss=controlled_explicit_miss,
    boundary_miss=response_diagnostics.boundary_miss,
    effective_miss=response_diagnostics.effective_miss,
    explicit_fake=controlled_explicit_fake,
    boundary_fake=response_diagnostics.boundary_fake,
    effective_fake=response_diagnostics.effective_fake,
    annotations=('Controlled sample only',),
    output=OUTPUT_DIR / f'{OUTPUT_TAG}_response_components.pdf',
    save_png=SAVE_PNG, grid=DRAW_GRID,
)
closure_canvas, studied_reco_to_truth, unfolded_studied_to_truth = (
    draw_unfolding_closure(
        studied_truth, studied_reco, unfolded_studied,
        target_label='Studied gen', target_role='gen',
        measured_label='Studied reco', unfolded_label='Unfolded reco',
        ratio_target_label='Gen',
        x_title='global #eta_{CM} bin', ratio_range=RATIO_Y_RANGE,
        annotations=(
            f'Studied sample, |#eta_{{CM}}| < {ETA_CUT:g}',
            f'Bayesian iterations: {N_ITERATIONS}',
        ),
        output=OUTPUT_DIR / f'{OUTPUT_TAG}_closure.pdf',
        save_png=SAVE_PNG, grid=DRAW_GRID,
        canvas_name='canvas_studied_closure',
        ratio_name_prefix='hStudiedClosure',
    )
)
display(response_canvas)
display(response_matrix_canvas)
display(response_components_canvas)
display(closure_canvas)

# Independent samples expose any difference between the two correction schemes.
method_comparison_canvas, method_comparison_ratios = draw_closure(
    {'Studied gen': studied_truth,
     'Factorized': unfolded_studied,
     'RooUnfold fakes/misses': unfolded_studied_inclusive_response},
    nominal='Studied gen', title='', x_title='global #eta_{CM} bin',
    y_title='Entries', ratio_range=RATIO_Y_RANGE, draw_nominal_ratio=False,
    annotations=(f'Studied sample, |#eta_{{CM}}| < {ETA_CUT:g}',
                 f'Controlled response, Bayesian iterations: {N_ITERATIONS}'),
    output=OUTPUT_DIR / f'{OUTPUT_TAG}_method_comparison.pdf',
    save_png=SAVE_PNG, grid=DRAW_GRID,
    canvas_name='canvas_studied_method_comparison',
)
factorized_to_studied_truth = method_comparison_ratios['Factorized']
factorized_to_studied_truth.SetName('hFactorizedToStudiedTruth')
inclusive_response_to_studied_truth = method_comparison_ratios['RooUnfold fakes/misses']
inclusive_response_to_studied_truth.SetName('hInclusiveResponseToStudiedTruth')
truth_closure_metrics = calculate_closure_metrics(
    unfolded_studied, studied_truth,
)
print('Independent truth closure:')
print(f'  integral ratio = {truth_closure_metrics.integral_ratio:.6f}')
print(f'  yield-weighted L1 difference = {truth_closure_metrics.relative_l1:.2%}')
print(f'  mean relative difference in {truth_closure_metrics.compared_bins} populated bins = ' 
      f'{truth_closure_metrics.mean_absolute_relative:.2%}')
method_comparison_canvas

## Studied closure in every pTave interval

Unlike the basics notebook, this is an independent closure test. Controlled and studied histograms contain different events, so their weighted bin fluctuations are different. Iterative Bayesian unfolding regularizes toward the controlled prior: increasing the iteration count reduces regularization and usually improves reconstructed-level refolding, but it can amplify fluctuations and worsen bin-by-bin truth closure. Neither ratio is required to equal exactly one.

In [ ]:
# Cell role: construct derived ratios, efficiencies, or correction factors.
# Interpretation: The numerator/denominator relationship determines whether independent or binomial errors are valid.
# The preceding Markdown gives the equations and physics assumptions for this step.
(
    unfolded_studied_by_pt,
    studied_reco_to_truth_by_pt,
    unfolded_studied_to_truth_by_pt,
    closure_canvases_by_pt,
) = draw_unfolding_closure_by_pt(
    unfolded_studied, studied_truth_by_pt, studied_reco_by_pt, layout,
    output_dir=OUTPUT_DIR, output_tag=OUTPUT_TAG,
    target_label='Studied Gen', target_role='gen',
    measured_label='Studied reco', ratio_target_label='Gen',
    eta_range=ETA_UNFOLDING_RANGE, ratio_range=RATIO_Y_RANGE,
    annotation_prefix=(
        'Independent studied sample',
        f'Bayesian iterations: {N_ITERATIONS}',
    ),
    save_png=SAVE_PNG, grid=DRAW_GRID,
)
(
    normalized_unfolded_to_controlled_by_pt,
    normalized_studied_reco_to_controlled_by_pt,
    normalized_unfolded_reco_to_controlled_by_pt,
    normalized_controlled_gen_comparison_canvases,
) = draw_unfolding_closure_by_pt(
    unfolded_studied, controlled_truth_by_pt, studied_reco_by_pt, layout,
    output_dir=OUTPUT_DIR, output_tag=OUTPUT_TAG,
    target_label='Controlled Gen', target_role='gen',
    measured_label='Studied reco', unfolded_label='Unfolded reco',
    ratio_target_label='Gen', normalize=True,
    output_suffix='normalized_to_controlled_gen',
    object_name_prefix='NormalizedToControlledGenPt',
    eta_range=ETA_UNFOLDING_RANGE, ratio_range=RATIO_Y_RANGE,
    annotation_prefix=(
        'Self-normalized shapes',
        f'Bayesian iterations: {N_ITERATIONS}',
    ),
    save_png=SAVE_PNG, grid=DRAW_GRID,
)
closure_canvases_by_pt, normalized_controlled_gen_comparison_canvases

## Forward-folded studied closure

Apply the inclusive controlled-sample response to the efficiency-corrected unfolded studied distribution. The response matrix reapplies efficiency and migration, producing matched reco. Fake fractions are then restored independently in every reconstructed bin using the controlled fake-to-matched-reco ratio—equivalently, the matched prediction is divided by controlled purity. No controlled/studied sample-size rescaling is applied.

Forward-folded closure asks whether the regularized truth estimate reproduces studied reco after the detector model; truth closure asks whether that estimate reproduces the independently generated studied truth. More iterations can improve the first while worsening the second. Comparisons are drawn in flattened global bins and separately in every $p_T^{ave}$ interval, with standard independent-error propagation in the displayed ratios.

In [ ]:
# Cell role: project multidimensional inputs into the configured analysis bins.
# Interpretation: Selections are applied before arithmetic so migrations and boundary bins remain explicit.
# The preceding Markdown gives the equations and physics assumptions for this step.
forward_fold_result = forward_fold_truth(
    inclusive_response_bundle, unfolded_studied, diagnostics=response_diagnostics,
    add_fakes=FORWARD_FOLD_ADD_FAKES, fake_normalization='matched_fraction',
    name='hForwardFoldedUnfoldedStudiedEtaCM',
)
forward_folded_studied = forward_fold_result.histogram
print('Fake normalization: controlled fake/matched-reco ratio in each measured bin')
print(f'ApplyToTruth included fakes: {forward_fold_result.apply_to_truth_includes_fakes}')

forward_folded_canvas, forward_folded_flattened_ratios = draw_closure(
    {'Studied reco': studied_reco, 'Forward-folded': forward_folded_studied},
    nominal='Studied reco', title='', x_title='global #eta_{CM} bin', y_title='Entries',
    ratio_range=RATIO_Y_RANGE, draw_nominal_ratio=False,
    annotations=(
        f'Studied sample, |#eta_{{CM}}| < {ETA_CUT:g}',
        f'Controlled response, Bayesian iterations: {N_ITERATIONS}',
    ),
    output=OUTPUT_DIR / f'{OUTPUT_TAG}_forward_folded_closure.pdf',
    save_png=SAVE_PNG, grid=DRAW_GRID,
    canvas_name='canvas_forward_folded_studied_closure',
)
forward_folded_studied_to_reco = forward_folded_flattened_ratios['Forward-folded']
forward_folded_studied_to_reco.SetName('hForwardFoldedStudiedToReco')
refold_closure_metrics = calculate_closure_metrics(
    forward_folded_studied, studied_reco,
)
print('Forward-folded reco closure:')
print(f'  integral ratio = {refold_closure_metrics.integral_ratio:.6f}')
print(f'  yield-weighted L1 difference = {refold_closure_metrics.relative_l1:.2%}')
print(f'  mean relative difference in {refold_closure_metrics.compared_bins} populated bins = ' 
      f'{refold_closure_metrics.mean_absolute_relative:.2%}')

forward_folded_studied_by_pt = unflatten_to_eta_projections(
    forward_folded_studied, studied_reco_by_pt, layout,
    name_prefix='hForwardFoldedStudiedEtaCM',
)
forward_folded_canvases_by_pt = []
forward_folded_studied_to_reco_by_pt = []
for pt_index, ((pt_low, pt_high), reco_hist, folded_hist) in enumerate(
    zip(PT_AVE_INTERVALS, studied_reco_by_pt, forward_folded_studied_by_pt)
):
    canvas, ratios = draw_closure(
        {'Studied reco': reco_hist, 'Forward-folded': folded_hist},
        nominal='Studied reco', title='', x_title='#eta_{CM}', y_title='Entries',
        ratio_range=RATIO_Y_RANGE, draw_nominal_ratio=False,
        x_range=ETA_UNFOLDING_RANGE,
        annotations=(
            'Controlled response applied to studied unfolding',
            f'{pt_low:g} < p_{{T}}^{{ave}} < {pt_high:g} GeV',
            f'Bayesian iterations: {N_ITERATIONS}',
        ),
        output=OUTPUT_DIR / f'{OUTPUT_TAG}_forward_folded_closure_pt_{pt_low:g}_{pt_high:g}.pdf',
        save_png=SAVE_PNG, grid=DRAW_GRID,
        canvas_name=f'canvas_forward_folded_studied_closure_pt{pt_index}',
    )
    ratio = ratios['Forward-folded']
    ratio.SetName(f'hForwardFoldedStudiedToReco_ptBin{pt_index}')
    forward_folded_canvases_by_pt.append(canvas)
    forward_folded_studied_to_reco_by_pt.append(ratio)

display(forward_folded_canvas)
forward_folded_canvases_by_pt

## Save unfolding artifacts


In [ ]:
# Cell role: construct derived ratios, efficiencies, or correction factors.
# Interpretation: The numerator/denominator relationship determines whether independent or binomial errors are valid.
# The preceding Markdown gives the equations and physics assumptions for this step.
output_histograms = (
    controlled_truth, controlled_reco, studied_truth, studied_reco,
    controlled_explicit_miss, controlled_explicit_fake, controlled_response,
    response_diagnostics.matched_truth, response_diagnostics.matched_measured,
    response_diagnostics.effective_miss, response_diagnostics.effective_fake,
    response_diagnostics.boundary_miss, response_diagnostics.boundary_fake,
    studied_reco_signal, controlled_reco_purity, controlled_truth_efficiency,
    unfolded_studied_matched, unfolded_studied,
    unfolded_studied_inclusive_response,
    studied_reco_to_truth, unfolded_studied_to_truth,
    factorized_to_studied_truth, inclusive_response_to_studied_truth,
    *controlled_truth_by_pt, *controlled_reco_by_pt,
    *studied_truth_by_pt, *studied_reco_by_pt,
    *controlled_explicit_miss_by_pt, *controlled_explicit_fake_by_pt,
    *normalized_original_shapes,
    *original_studied_to_controlled_ratios,
    *unfolded_studied_by_pt, *studied_reco_to_truth_by_pt,
    *unfolded_studied_to_truth_by_pt,
    *normalized_unfolded_to_controlled_by_pt,
    *normalized_studied_reco_to_controlled_by_pt,
    *normalized_unfolded_reco_to_controlled_by_pt,
    forward_fold_result.folded_signal, forward_fold_result.fake,
    forward_folded_studied, forward_folded_studied_to_reco,
    *forward_folded_studied_by_pt, *forward_folded_studied_to_reco_by_pt,
)
write_unfolding_output(
    OUTPUT_ROOT_FILE, histograms=output_histograms, covariance=covariance,
    covariance_name='hUnfoldedStudiedCovariance',
    response=matched_response_bundle.response,
    response_name='controlledMatchedResponseEtaCM',
    metadata={
        'generator': GENERATOR, 'direction': DIRECTION, 'eta_cut': ETA_CUT,
        'eta_unfolding_range': ETA_UNFOLDING_RANGE,
        'pt_ave_bins': PT_AVE_INTERVALS, 'ptave_bin_set': PTAVE_BIN_SET,
        'iterations': N_ITERATIONS, 'response_scale': RESPONSE_SCALE,
        'split': 'controlled60_studied40',
        'controlled_truth_fraction': controlled_truth_fraction,
        'controlled_reco_fraction': controlled_reco_fraction,
        'unfolding_method': 'factorized_controlled_purity_migration_efficiency',
        'require_fakes_reference': REQUIRE_FAKES,
        'handle_fakes_reference': HANDLE_FAKES,
        'forward_fold_add_fakes': FORWARD_FOLD_ADD_FAKES,
        'forward_fold_fake_normalization': 'controlled_binwise_fake_to_matched_reco',
        'apply_to_truth_includes_fakes': forward_fold_result.apply_to_truth_includes_fakes,
        'response_sparsity': response_accounting.response_sparsity,
        'global_efficiency': response_accounting.global_efficiency,
        'global_fake_fraction': response_accounting.global_fake_fraction,
        'truth_closure_relative_l1': truth_closure_metrics.relative_l1,
        'refold_closure_relative_l1': refold_closure_metrics.relative_l1,
    },
)
print(f'Wrote {OUTPUT_ROOT_FILE}')